# ACT 학습 (Colab)
런타임 > 런타임 유형 변경 > **T4 GPU** 선택 후 위에서부터 순서대로 실행. 설정은 **셀 2**만 수정.
- 준비: 노트북에서 `dev/outputs/<이름>.zip` (예: `pick_tape.zip`)을 드라이브 `MyDrive/act/`에 업로드
- 셀 1 설치 후 "세션 다시 시작" 안내가 뜨면 다시 시작하고 **셀 1부터** 다시 실행
- **세션이 끊기면 그냥 위에서부터 다시 전부 실행** → 드라이브의 마지막 체크포인트부터 이어서 학습

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "lerobot[transformers-dep]==0.4.4"  # 코랩 기본 transformers(5.x)는 lerobot이 까는 huggingface_hub 0.35와 충돌 → 맞는 버전으로 같이 설치
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ===== 여기만 수정 =====
NAME          = "pick_tape"                                # 데이터셋 이름 (zip 이름과 같게)
DATASET_ZIP   = f"/content/drive/MyDrive/act/{NAME}.zip"   # 노트북에서 압축한 데이터셋
SAVE_DIR      = f"/content/drive/MyDrive/act/act_{NAME}"   # 체크포인트를 저장할 드라이브 폴더
STEPS         = 50000                                      # 총 학습 스텝
SAVE_EVERY    = 10000                                      # 이 스텝마다 드라이브에 백업 (세션이 끊겨도 여기서부터 이어감)
BATCH         = 8
VIDEO_BACKEND = "torchcodec"                               # 영상 읽기(decode) 에러가 나면 "pyav"
# ========================
import glob, os
DS, OUT = f"/content/ds/{NAME}", f"/content/train/act_{NAME}"
if not os.path.isdir(DS):
    !mkdir -p /content/ds && unzip -oq "{DATASET_ZIP}" -d /content/ds
assert os.path.isfile(f"{DS}/meta/info.json"), f"{DS}/meta/info.json 없음: zip 안에 {NAME}/ 폴더가 있어야 함"

# 드라이브에 체크포인트가 있으면 로컬로 가져와서 이어서 학습
saved = sorted(glob.glob(f"{SAVE_DIR}/checkpoints/[0-9]*"))
if not os.path.exists(f"{OUT}/checkpoints/last"):
    !rm -rf "{OUT}"  # 첫 저장 전에 끊긴 찌꺼기 (있으면 lerobot이 FileExistsError)
    if saved:
        latest = saved[-1]
        !mkdir -p "{OUT}/checkpoints" && cp -r "{latest}" "{OUT}/checkpoints/"
        os.symlink(os.path.basename(saved[-1]), f"{OUT}/checkpoints/last")
done = int(os.path.basename(os.path.realpath(f"{OUT}/checkpoints/last"))) if os.path.exists(f"{OUT}/checkpoints/last") else 0
print("이어서 시작할 스텝:", done)

In [ ]:
# (선택) 20스텝만 돌려서 속도 확인: 끝에 나오는 s/step × STEPS = 예상 학습 시간
!rm -rf /content/smoke && lerobot-train --policy.type=act --dataset.repo_id=local/{NAME} --dataset.root={DS} --dataset.video_backend={VIDEO_BACKEND} --output_dir=/content/smoke --policy.device=cuda --policy.push_to_hub=false --steps=20 --batch_size={BATCH} --save_checkpoint=false --wandb.enable=false

In [ ]:
# SAVE_EVERY 스텝씩 끊어서 학습 → 끝날 때마다 드라이브로 복사
# 드라이브에서는 lerobot이 checkpoints/last 심볼릭 링크를 못 만들어서, 학습은 로컬 디스크에서 하고 복사만 함
# 체크포인트 1개 ≈ 600MB (모델 200MB + 이어하기용 training_state 400MB). training_state는 최신 것만 드라이브에 남김
for target in range(done + SAVE_EVERY, STEPS + 1, SAVE_EVERY):
    if not os.path.exists(f"{OUT}/checkpoints/last"):  # 처음 시작
        !lerobot-train --policy.type=act --dataset.repo_id=local/{NAME} --dataset.root={DS} --dataset.video_backend={VIDEO_BACKEND} --output_dir={OUT} --policy.device=cuda --policy.push_to_hub=false --steps={target} --save_freq={SAVE_EVERY} --batch_size={BATCH} --wandb.enable=false
    else:  # 이어하기: 설정은 train_config.json 그대로, 목표 스텝만 늘림
        !lerobot-train --config_path={OUT}/checkpoints/last/pretrained_model/train_config.json --resume=true --steps={target}
    ckpt, prev = f"{OUT}/checkpoints/{target:06d}", f"{SAVE_DIR}/checkpoints/{target - SAVE_EVERY:06d}/training_state"
    assert os.path.isdir(ckpt), f"{ckpt} 없음: 위 로그에서 에러 확인"
    !mkdir -p "{SAVE_DIR}/checkpoints" && cp -r "{ckpt}" "{SAVE_DIR}/checkpoints/" && rm -rf "{prev}"
    print(f"===== {target} 스텝 드라이브 저장 완료 =====")

## 노트북(로봇)으로 가져오기
드라이브 `act/act_pick_tape/checkpoints/050000/pretrained_model` 폴더를 다운로드해서
노트북 `dev/outputs/train/act_pick_tape/checkpoints/last/pretrained_model` 에 넣고 `python scripts/run_auto.py`.
중간 체크포인트(예: `020000`)도 같은 자리에 넣으면 학습이 끝나기 전에 미리 시험해볼 수 있음.

다른 물체(`pick_bottle`, `pick_pen`)는 zip을 올리고 셀 2의 `NAME`만 바꿔서 다시 실행.